In [ ]:
import numpy as np
import tensorflow as tf
import os

# Path to the newly created model
model_path = 'app/src/main/assets/vsr_lora_model.tflite'
checkpoint_dir = '/content/drive/MyDrive/LipertyData/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Loading model from: {model_path}")

# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

train_func = interpreter.get_signature_runner('train')

# Prepare data matching the fixed shapes
dummy_video = np.random.randn(1, 50, 88, 88, 1).astype(np.float32)
# Providing One-Hot labels: [Batch=1, Time=50, Classes=40]
dummy_labels = np.zeros((1, 50, 40), dtype=np.float32)
for i in range(50):
    idx = np.random.randint(0, 40)
    dummy_labels[0, i, idx] = 1.0

print("Executing a training step with One-Hot labels...")
try:
    result = train_func(
        train_train_video_input=dummy_video,
        train_train_target_labels=dummy_labels
    )
    print(f"Training step successful! Loss: {result['loss']}")
    print(f"Checkpoints will be stored in: {checkpoint_dir}")
except Exception as e:
    print(f"Error: {e}")

ValueError: JAX requires ml_dtypes version 0.5 or newer; installed version is 0.3.2.

In [ ]:
import tensorflow as tf
import numpy as np

model_path = 'app/src/main/assets/vsr_lora_model.tflite'
interpreter = tf.lite.Interpreter(model_path=model_path)

print("--- Input Details ---")
input_details = interpreter.get_input_details()
for i, detail in enumerate(input_details):
    print(f"Index: {detail['index']}, Name: {detail['name']}, Shape: {detail['shape']}, Shape Signature: {detail['shape_signature']}, DType: {detail['dtype']}")

print("\n--- Signature Details ---")
signatures = interpreter.get_signature_list()
print(signatures)

In [ ]:
import tensorflow as tf

model_path = 'app/src/main/assets/vsr_lora_model.tflite'
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Inspect all signatures
signatures = interpreter.get_signature_list()
print("Available signatures:", signatures)

# Detailed inspection of the 'train' signature
train_details = interpreter.get_signature_runner('train').get_full_signature_list()
print("\nDetailed 'train' signature inputs:")
for name, detail in train_details['inputs'].items():
    print(f"- Input Name: {name}, Shape: {detail['shape']}, Type: {detail['dtype']}")

print("\nDetailed 'train' signature outputs:")
for name, detail in train_details['outputs'].items():
    print(f"- Output Name: {name}, Shape: {detail['shape']}, Type: {detail['dtype']}")

In [ ]:
import os
# Workaround for the __dict__ descriptor error in TF + Py3.12
os.environ['TF_USE_LEGACY_KERAS'] = '1'

!mkdir -p data/checkpoints
!python tools/create_trainable_model.py
print("Ready for training.")

In [ ]:
!pip install --upgrade "ml_dtypes>=0.5.0"
import ml_dtypes
print(f'Installed ml_dtypes version: {ml_dtypes.__version__}')

In [ ]:
import os

# Patch the create_trainable_model.py script to force static dimensions throughout the architecture
with open('/content/Liperty/tools/create_trainable_model.py', 'r') as f:
    content = f.read()

# Fix 1: Force static batch and sequence length in the model input
content = content.replace(
    'input_video = tf.keras.layers.Input(shape=(50, 88, 88, 1)',
    'input_video = tf.keras.layers.Input(batch_size=1, shape=(50, 88, 88, 1)'
)

# Fix 2: Ensure the training signature is strictly defined with names and static shapes
content = content.replace(
    'def train(self, video_input, target_labels):',
    '# Explicitly defined signature inputs\n    @tf.function(input_signature=[\n        tf.TensorSpec(shape=[1, 50, 88, 88, 1], dtype=tf.float32, name="train_video_input"),\n        tf.TensorSpec(shape=[1, 50], dtype=tf.float32, name="train_target_labels")\n    ])\n    def train(self, train_video_input, train_target_labels):'
)

# Map variable names
content = content.replace('video_input', 'train_video_input')
content = content.replace('target_labels', 'train_target_labels')

# Ensure optimizer tracking
if 'model.optimizer = optimizer' not in content:
    content = content.replace(
        'tf.saved_model.save(',
        'model.optimizer = optimizer\n    tf.saved_model.save('
)

with open('/content/Liperty/tools/create_trainable_model.py', 'w') as f:
    f.write(content)

print("Script patched with full static architecture and unique signature names.")

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!python tools/create_trainable_model.py
print("Ready for training.")

# Liperty VSR Training Notebook (Parallel & GDrive Persistent)
This version of the notebook is optimized for **speed** (simultaneous downloads) and **persistence** (Google Drive storage).

**Features:**
- Uses `aria2c` for high-speed parallel downloads.
- Persistent storage on Google Drive (`MyDrive/LipertyData`).
- Automated setup and LoRA initialization.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Define Persistence Paths
import os
gdrive_data_root = '/content/drive/MyDrive/LipertyData'
!mkdir -p {gdrive_data_root}/LRS2-2Mix
!mkdir -p {gdrive_data_root}/VVAD-LRS3

# 3. Clone Repository
repo_dir = '/content/Liperty'
if not os.path.exists(repo_dir):
    !git clone https://github.com/HereLiesAz/Liperty.git {repo_dir}

%cd {repo_dir}

# 4. Force Symlink to GDrive
!rm -rf /content/Liperty/data
!ln -s {gdrive_data_root} /content/Liperty/data

# 5. Install Dependencies (Python 3.12 Optimized)
!apt-get install -y aria2
!pip install --upgrade pip
!pip install "ml_dtypes>=0.5.0" "numpy<2.0.0"
!pip install datasets transformers mediapipe opencv-python onnx==1.16.1 tensorflow==2.16.1 torch

# Use onnx2tf instead of the broken onnx-tf
!pip install onnx2tf sagemaker-onnx
!pip install onnx_graphsurgeon --index-url https://pypi.ngc.nvidia.com

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Liperty
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
  Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (5.0 MB)
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tf

  Using cached ml_dtypes-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached ml_dtypes-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.4
    Uninstalling ml_dtypes-0.5.4:
      Successfully uninstalled ml_dtypes-0.5.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.16.1 which is incompatible.
tensorstore 0.1.81 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.3.2 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.16.1 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.16.1 which is incompatible.
jax 0.7.2 requires ml_dtyp

ERROR: Could not find a version that satisfies the requirement sagemaker-onnx (from versions: none)
ERROR: No matching distribution found for sagemaker-onnx
Looking in indexes: https://pypi.ngc.nvidia.com


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## High-Speed Parallel Data Preparation
The following cell downloads all specified datasets simultaneously to your Google Drive.

In [ ]:
import os

def check_status(path, name):
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        print(f'[+] {name}: Found {path} ({size_gb:.2f} GB)')
    else:
        print(f'[!] {name}: File {path} not found yet.')

print('Checking current download sizes...')
check_status('data/LRS2-2Mix/lrs2.tar.gz', 'LRS2-2Mix')
check_status('data/VVAD-LRS3/vvadlrs3.zip', 'VVAD-LRS3')

In [ ]:
import os
import json
from google.colab import userdata

# Securely fetch credentials from Colab Secrets
try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')

    if kaggle_username and kaggle_key:
        dot_kaggle = os.path.expanduser('~/.kaggle')
        os.makedirs(dot_kaggle, exist_ok=True)
        with open(os.path.join(dot_kaggle, 'kaggle.json'), 'w') as f:
            json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
        !chmod 600 ~/.kaggle/kaggle.json
        print("Kaggle credentials configured successfully from Secrets!")
    else:
        print("Credentials not found in Secrets. Please add KAGGLE_USERNAME and KAGGLE_KEY to the secrets tab (🔑).")
except Exception as e:
    print(f"Error accessing secrets: {e}. Please ensure you have set up the secrets correctly.")

In [ ]:
import os

# Ensure we are using the GDrive paths
lrs2_path = "/content/drive/MyDrive/LipertyData/LRS2-2Mix"
vvad_path = "/content/drive/MyDrive/LipertyData/VVAD-LRS3"
!mkdir -p {lrs2_path} {vvad_path}

print("Starting simultaneous downloads directly to Google Drive...")

# Download LRS2-2Mix with aria2c
lrs2_url = "https://huggingface.co/datasets/JusperLee/LRS2-2Mix/resolve/main/lrs2.tar.gz"
!aria2c -c -x 16 -s 16 -d {lrs2_path} -o lrs2.tar.gz {lrs2_url} &

# Download VVAD-LRS3 using aria2c (Direct HuggingFace link for VVAD-LRS3 setup)
vvad_url = "https://huggingface.co/datasets/adrianlubitz/vvadlrs3/resolve/main/vvadlrs3.zip"
!aria2c -c -x 16 -s 16 -d {vvad_path} -o vvadlrs3.zip {vvad_url} &

!wait

print("\nDownloads finished. Checking files...")
!ls -lh {lrs2_path}
!ls -lh {vvad_path}

## Training & Fine-Tuning

In [ ]:
import ml_dtypes
print(f'Currently loaded ml_dtypes version: {ml_dtypes.__version__}')
if ml_dtypes.__version__ < '0.5.0':
    print('\n!!! STILL OLD VERSION. Please click Restart session in the popup or go to Runtime > Restart session !!!')
else:
    print('\nSuccess! Version is compatible. You can run the training cell now.')

In [ ]:
import os
# Workaround for the __dict__ descriptor error in TF + Py3.12
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# Ensure we are in the repo directory
repo_path = '/content/Liperty'
if os.path.exists(repo_path):
    os.chdir(repo_path)

!mkdir -p data/checkpoints
!python /content/Liperty/tools/create_trainable_model.py
print("Ready for training.")

In [ ]:
import numpy as np
import tensorflow as tf
import os

# Path to the newly created model
model_path = 'app/src/main/assets/vsr_lora_model.tflite'
checkpoint_dir = '/content/drive/MyDrive/LipertyData/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Loading model from: {model_path}")

# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Get the signature runner for 'train'
train_func = interpreter.get_signature_runner('train')

# Video: [1, 50, 88, 88, 1]
dummy_video = np.random.randn(1, 50, 88, 88, 1).astype(np.float32)

# One-Hot labels: [1, 50, 40] as required by the model signature
dummy_labels = np.zeros((1, 50, 40), dtype=np.float32)
for i in range(50):
    idx = np.random.randint(0, 40)
    dummy_labels[0, i, idx] = 1.0

print("Executing a training step with One-Hot labels and corrected names...")
try:
    result = train_func(
        train_train_video_input=dummy_video,
        train_train_target_labels=dummy_labels
    )
    print(f"Training step successful! Loss: {result['loss']}")
    print(f"Checkpoints will be stored in: {checkpoint_dir}")
except Exception as e:
    print(f"Training attempt failed: {e}")

In [ ]:
import tensorflow as tf

model_path = 'app/src/main/assets/vsr_lora_model.tflite'
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

train_runner = interpreter.get_signature_runner('train')
input_details = train_runner.get_input_details()
output_details = train_runner.get_output_details()

print('--- Signature Input Details ---')
for name, detail in input_details.items():
    print(f'Name: {name}, Index: {detail["index"]}, Shape: {detail["shape"]}')

print('\n--- Tensors near Node 58 ---')
all_details = interpreter.get_tensor_details()
for d in all_details:
    if 50 <= d['index'] <= 65:
        print(f'Index: {d["index"]}, Name: {d["name"]}, Shape: {d["shape"]}')

In [ ]:
import tensorflow as tf

model_path = 'app/src/main/assets/vsr_lora_model.tflite'
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

print("--- Current Training Signature Details ---")
signatures = interpreter.get_signature_list()
if 'train' in signatures:
    print(f"Signature 'train' expects: {signatures['train']}")
else:
    print(f"'train' signature not found. Available: {list(signatures.keys())}")

In [ ]:
import tensorflow as tf

model_path = 'app/src/main/assets/vsr_lora_model.tflite'
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

print("--- Detailed Tensor Map ---")
details = interpreter.get_tensor_details()
for d in details:
    # Focusing on tensors around the error index if possible, or looking for shapes of size 50
    if 50 in d['shape'] or d['index'] == 58:
        print(f"Index: {d['index']}, Name: {d['name']}, Shape: {d['shape']}, DType: {d['dtype']}")

# Also check the signature runner's specific input/output tensor indices
train_runner = interpreter.get_signature_runner('train')
print("\n--- Train Signature Runner Details ---")
print(f"Inputs: {train_runner._inputs}")
print(f"Outputs: {train_runner._outputs}")

## Export to TFLite

In [ ]:
import os
# Ensure dependencies for conversion and VALLR scripts are present
!pip install "ml_dtypes>=0.5.0" "numpy<2.0.0" onnx2tf onnx configargparse

os.environ['TF_USE_LEGACY_KERAS'] = '1'

print("Running VALLR TFLite conversion script...")
# Change directory to VALLR to ensure internal imports work
%cd /content/Liperty/VALLR
!python convert_to_tflite.py
%cd /content/Liperty

Running VALLR TFLite conversion script...
/content/Liperty/VALLR
2026-03-17 21:27:45.141817: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773782865.163461   28449 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773782865.170429   28449 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773782865.188669   28449 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773782865.188698   28449 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773782865

In [ ]:
!ls -lh /content/Liperty/app/src/main/assets/vsr_lora_model.tflite
!ls -lh /content/Liperty/app/src/main/assets/vallr_model.tflite

-rw-r--r-- 1 root root 88M Mar 17 21:17 /content/Liperty/app/src/main/assets/vsr_lora_model.tflite
-rw-r--r-- 1 root root 349M Mar 17 22:04 /content/Liperty/app/src/main/assets/vallr_model.tflite


In [ ]:
import os
print("Contents of /content/Liperty/VALLR:")
!ls -laR /content/Liperty/VALLR

print("\nChecking if onnx2tf left any conversion logs:")
!ls -la /content/Liperty/*.log

Contents of /content/Liperty/VALLR:
/content/Liperty/VALLR:
total 358296
drwxr-xr-x 5 root root      4096 Mar 17 21:31 .
drwxr-xr-x 9 root root      4096 Mar 17 21:22 ..
-rw-r--r-- 1 root root      7425 Mar 17 20:10 config.py
-rw-r--r-- 1 root root     11104 Mar 17 20:10 convert_to_tflite.py
drwxr-xr-x 3 root root      4096 Mar 17 20:10 Data
-rw-r--r-- 1 root root     49364 Mar 17 20:10 face_cropper.py
-rw-r--r-- 1 root root     22440 Mar 17 20:10 main.py
-rw-r--r-- 1 root root   1196356 Mar 17 21:51 model.onnx
-rw-r--r-- 1 root root 365559808 Mar 17 21:51 model.onnx.data
drwxr-xr-x 3 root root      4096 Mar 17 21:26 Models
drwxr-xr-x 2 root root      4096 Mar 17 21:26 __pycache__
-rw-r--r-- 1 root root      5776 Mar 17 20:10 README.md
-rw-r--r-- 1 root root      1940 Mar 17 20:10 requirements.txt

/content/Liperty/VALLR/Data:
total 40
drwxr-xr-x 3 root root 4096 Mar 17 20:10 .
drwxr-xr-x 5 root root 4096 Mar 17 21:31 ..
-rw-r--r-- 1 root root 4320 Mar 17 20:10 dataset.py
-rw-r--r-- 1 

In [ ]:
import os
print("Searching for any generated .tflite files in the project...")
!find /content/Liperty -name "*.tflite"

# Also check the specific assets folder again to see if it's a naming mismatch
print("\nContents of assets folder:")
!ls -lh /content/Liperty/app/src/main/assets/

Searching for any generated .tflite files in the project...
/content/Liperty/app/src/main/assets/voice_converter.tflite
/content/Liperty/app/src/main/assets/tramba_model.tflite
/content/Liperty/app/src/main/assets/vsr_lora_model.tflite
/content/Liperty/app/src/main/assets/vsr_model.tflite
/content/Liperty/app/src/main/assets/ssr_model.tflite

Contents of assets folder:
total 91M
-rw-r--r-- 1 root root 3.6M Mar 17 20:10 face_landmarker.task
-rw-r--r-- 1 root root 3.0K Mar 17 20:10 homophones.json
-rw-r--r-- 1 root root  29K Mar 17 20:10 ssr_model.tflite
-rw-r--r-- 1 root root  29K Mar 17 20:10 tramba_model.tflite
-rw-r--r-- 1 root root  29K Mar 17 20:10 voice_converter.tflite
-rw-r--r-- 1 root root  88M Mar 17 21:17 vsr_lora_model.tflite
-rw-r--r-- 1 root root  29K Mar 17 20:10 vsr_model.tflite


In [ ]:
import os
print("Searching for external weight files (.data or .bin) in VALLR and assets...")
!find /content/Liperty -name "*.data" -o -name "*.bin"

# Also check if onnx2tf created a subfolder with the real model
!ls -R /content/Liperty/app/src/main/assets/

Searching for external weight files (.data or .bin) in VALLR and assets...
/content/Liperty/VALLR/model.onnx.data
/content/Liperty/app/src/main/assets/:
face_landmarker.task  tramba_model.tflite     vsr_model.tflite
homophones.json       voice_converter.tflite
ssr_model.tflite      vsr_lora_model.tflite


In [ ]:
import os
tflite_path = '/content/Liperty/app/src/main/assets/vallr_model.tflite'
if os.path.exists(tflite_path):
    if os.path.isfile(tflite_path):
        size = os.path.getsize(tflite_path) / (1024 * 1024)
        if size > 100.0:
            print(f'Success! VALLR TFLite model found at {tflite_path}')
            print(f'Final Model Size: {size:.2f} MB')
        else:
            print(f'Warning: Model file is still small ({size:.4f} MB). Checking for separate variable files...')
            !find /content/Liperty/VALLR -name "*variables*"
    else:
        print(f'Error: {tflite_path} is a directory.')
else:
    print('TFLite model not found in assets.')

Success! VALLR TFLite model found at /content/Liperty/app/src/main/assets/vallr_model.tflite
Final Model Size: 348.54 MB


In [ ]:
import os
onnx_path = '/content/Liperty/VALLR/model.onnx'
if os.path.exists(onnx_path):
    size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f'ONNX file found: {onnx_path}')
    print(f'ONNX Size: {size:.4f} MB')
else:
    print('ONNX file missing entirely.')

ONNX file found: /content/Liperty/VALLR/model.onnx
ONNX Size: 348.4732 MB


In [ ]:
import os
tflite_path = '/content/Liperty/app/src/main/assets/vallr_model.tflite'
if os.path.exists(tflite_path):
    size = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f'Success! VALLR TFLite model found at {tflite_path}')
    print(f'Model Size: {size:.2f} MB')
else:
    print('TFLite model not found. Checking if the ONNX file exists for manual conversion...')
    if os.path.exists('/content/Liperty/VALLR/model.onnx'):
        print('ONNX file exists. Attempting final manual conversion...')
        !onnx2tf -i /content/Liperty/VALLR/model.onnx -o {tflite_path} --non_verbose
    else:
        print('ONNX file was not generated.')

Success! VALLR TFLite model found at /content/Liperty/app/src/main/assets/vallr_model.tflite
Model Size: 0.00 MB


In [ ]:
with open('/content/Liperty/VALLR/Models/VALLR.py', 'r') as f:
    print(f.read())

import torch
import torch.nn as nn
import torch.nn.init as init
from transformers import VideoMAEConfig, Wav2Vec2Config, Wav2Vec2ForCTC, VideoMAEModel

class VALLR(nn.Module):
    def __init__(self, videomae_config: VideoMAEConfig, wav2vec_config: Wav2Vec2Config, adapter_dim: int):
        super(VALLR, self).__init__()

        # Initialize the VideoMAE model for feature extraction
        self.videomae = VideoMAEModel(videomae_config) 

        # VideoMAE feature size
        videomae_feature_size = videomae_config.hidden_size  # Typically 768 for VideoMAE

        # Downsample the time dimension using multiple Conv1D and Pooling layers
        self.downsampling = nn.Sequential(
            nn.Conv1d(in_channels=videomae_feature_size, out_channels=adapter_dim, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(adapter_dim, eps=1e-5, momentum=0.1, affine=True),
            nn.ReLU(),

            nn.Conv1d(in_channels=adapter_dim, out_channels=adapter_dim, kernel_size=3, s

In [ ]:
import torch
import tensorflow as tf
import sys
import os
from transformers import VideoMAEConfig, Wav2Vec2Config

# Add VALLR paths to sys
sys.path.append('/content/Liperty/VALLR')
sys.path.append('/content/Liperty/VALLR/Models')

from VALLR import VALLR

def export_vallr_v2():
    print("Initializing VALLR model...")
    v_conf = VideoMAEConfig(image_size=88, num_channels=3, num_frames=50, hidden_size=768)
    w_conf = Wav2Vec2Config()
    model = VALLR(videomae_config=v_conf, wav2vec_config=w_conf, adapter_dim=512)
    model.eval()

    dummy_input = torch.randn(1, 50, 3, 88, 88)
    onnx_path = "/content/Liperty/VALLR/model.onnx"
    target_file = "/content/Liperty/app/src/main/assets/vallr_model.tflite"

    print("Step 1: Exporting to ONNX...")
    torch.onnx.export(model, dummy_input, onnx_path, opset_version=11, input_names=['video_input'], output_names=['output'])

    print("Step 2: Converting ONNX to intermediate SavedModel folder...")
    saved_model_dir = "/content/Liperty/VALLR/intermediate_tf"
    if os.path.exists(saved_model_dir): import shutil; shutil.rmtree(saved_model_dir)
    !onnx2tf -i {onnx_path} -o {saved_model_dir} --non_verbose

    print("Step 3: Using TFLiteConverter API to bundle weights...")
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
    tflite_model = converter.convert()

    with open(target_file, 'wb') as f:
        f.write(tflite_model)

    size = os.path.getsize(target_file)/(1024*1024)
    print(f"Success! Final Model Size: {size:.2f} MB")

try:
    export_vallr_v2()
except Exception as e:
    print(f"Export Error: {e}")

Initializing VALLR model...


W0317 21:55:14.665000 27185 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Step 1: Exporting to ONNX...


W0317 21:55:15.152000 27185 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0317 21:55:15.153000 27185 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0317 21:55:15.156000 27185 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0317 21:55:15.158000 27185 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `VALLR([...]` with `torch.export.export(..., strict=False)`...
Video features shape: torch.Size([1, 625, 768])
Downsampled features shape: torch.Size([1, 512, 3])
[torch.onnx] Obtain model graph for `VALLR([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/no_previous_version.h:26: adapt: Assertion `

Applied 13 of general pattern rewrite rules.
Step 2: Converting ONNX to intermediate SavedModel folder...
Saved artifact at '/content/Liperty/VALLR/intermediate_tf'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 3, 88, 88, 50), dtype=tf.float32, name='video_input')
Output Type:
  List[TensorSpec(shape=(1, 3, 32), dtype=tf.float32, name=None), TensorSpec(shape=(1, 3, 768), dtype=tf.float32, name=None)]
^C
Step 3: Using TFLiteConverter API to bundle weights...
Success! Final Model Size: 348.54 MB


In [ ]:
with open('/content/Liperty/tools/convert_vallr.py', 'r') as f:
    print(f.read())

import torch
import torch.onnx
import tensorflow as tf
import onnx
from onnx_tf.backend import prepare
import os

def convert_vallr_to_tflite(pytorch_model_path, output_tflite_path):
    """
    Converts a VALLR PyTorch model to TFLite format.

    This is a template script. You will need the actual VALLR model definition
    and weights to run this successfully.
    """
    print(f"Converting {pytorch_model_path} to {output_tflite_path}...")

    # 1. Load PyTorch Model
    # You need to import the VALLR model class here.
    # from model import VALLR
    # model = VALLR()
    # model.load_state_dict(torch.load(pytorch_model_path, map_location=torch.device('cpu')))
    # model.eval()

    # Dummy placeholder for the model (Replace with actual model loading)
    class DummyModel(torch.nn.Module):
        def forward(self, x):
            return x

    model = DummyModel()
    print("Model loaded (Dummy placeholder). Replace with actual VALLR model loading.")

    # 2. Define Input Shap

In [ ]:
import os
# Check if the model needs to be downloaded or generated from the .py files
%cd /content/Liperty/VALLR
print("Checking for model file...")
if not os.path.exists('model.onnx'):
    print("model.onnx not found. Looking for an export script...")
    # Attempt to find a script that exports the VALLR model to ONNX
    !find . -name "*export*" -o -name "*save*"
else:
    print("model.onnx found. Proceeding with conversion check.")
%cd /content/Liperty

/content/Liperty/VALLR
Checking for model file...
model.onnx not found. Looking for an export script...
/content/Liperty


In [ ]:
!find /content/Liperty -name "*convert*" -o -name "*export*"

/content/Liperty/app/src/main/cpp/libs/opencv/sdk/native/jni/include/opencv2/gapi/own/exports.hpp
/content/Liperty/app/src/main/cpp/libs/opencv/sdk/native/jni/include/opencv2/gapi/own/convert.hpp
/content/Liperty/app/src/main/assets/voice_converter.tflite
/content/Liperty/tools/convert_vallr.py
/content/Liperty/VALLR/convert_to_tflite.py


In [ ]:
!ls -lh /content/Liperty/VALLR/Models

total 32K
-rw-r--r-- 1 root root 9.7K Mar 17 20:10 Llama.py
-rw-r--r-- 1 root root  11K Mar 17 20:10 ML_VALLR.py
-rw-r--r-- 1 root root 4.3K Mar 17 20:10 VALLR.py


In [ ]:
import os
!ls -R /content/Liperty | grep -E 'model.onnx|VALLR'

VALLR
/content/Liperty/VALLR:
/content/Liperty/VALLR/Data:
/content/Liperty/VALLR/Data/__pycache__:
/content/Liperty/VALLR/Models:
ML_VALLR.py
VALLR.py
